In [10]:
! pip install requests beautifulsoup4

  Using cached beautifulsoup4-4.12.3-py3-none-any.whl (147 kB)


In [37]:
def get_wikipedia_english_version(article_title):
    # Set up the API endpoint and parameters
    url = "https://he.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "titles": article_title,
        "prop": "langlinks",
        "lllimit": "max"  # Get all language links
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        return f"Error fetching data. Response code: {response.status_code}"
    
    data = response.json()
    
    pages = data['query']['pages']
    for page_id, page_info in pages.items():
        if 'langlinks' in page_info:
            langlinks = page_info['langlinks']
            languages = {lang['lang']: lang['*'] for lang in langlinks}
            return languages.get('en', None)
        else:
            return None

'Naomi Shemer'

In [55]:
import requests
import re
def parse_birth_date_heb(data):
    hebrew_months = {"בינואר": 1, "בפברואר": 2, "במרץ": 3, "באפריל": 4, "במאי": 5, "ביוני": 6, "ביולי": 7, "באוגוסט": 8, "בספטמבר": 9, "באוקטובר": 10, "בנובמבר": 11, "בדצמבר": 12}
    birth_date_match = re.search(r'"תאריך לידה":{.*?\[\[(\d{1,2}) (\w+)\]\] \[\[(\d{4})\]\]"}', data)
    if birth_date_match:
        birth_day, birth_month, birth_year = int(birth_date_match.group(1)), hebrew_months[birth_date_match.group(2)], int(birth_date_match.group(3))
        return (birth_year, birth_month, birth_day)
    return None

def parse_death_date_heb(data):
    hebrew_months = {"בינואר": 1, "בפברואר": 2, "במרץ": 3, "באפריל": 4, "במאי": 5, "ביוני": 6, "ביולי": 7, "באוגוסט": 8, "בספטמבר": 9, "באוקטובר": 10, "בנובמבר": 11, "בדצמבר": 12}
    death_date_match = re.search(r'"תאריך פטירה":{.*?\[\[(\d{1,2}) (\w+)\]\] \[\[(\d{4})\]\]"}', data)
    if death_date_match:
        death_day, death_month, death_year = int(death_date_match.group(1)), hebrew_months[death_date_match.group(2)], int(death_date_match.group(3))
        return (death_year, death_month, death_day)
    return None

#"birth_date":{"wt":"{{birth date|mf=yes|1930|7|13}}"},"death_date":{"wt":"{{death date and age|2004|6|26|1930|7|13|mf=y}}"}
def parse_birth_date_eng(data):
    birth_date_match = re.search(r'"birth_date":{.*?\|(\d{4})\|(\d{1,2})\|(\d{1,2})', data)
    if birth_date_match:
        birth_year, birth_month, birth_day = int(birth_date_match.group(1)), int(birth_date_match.group(2)), int(birth_date_match.group(3))
        return (birth_year, birth_month, birth_day)

def parse_death_date_eng(data):
    death_date_match = re.search(r'"death_date":{.*?\|(\d{4})\|(\d{1,2})\|(\d{1,2})', data)
    if death_date_match:
        death_year, death_month, death_day = int(death_date_match.group(1)), int(death_date_match.group(2)), int(death_date_match.group(3))
        return (death_year, death_month, death_day)

def get_birth_death_dates(name):
    # Format the name to be used in the Wikipedia API
    query = name.replace(" ", "_")
    heb_url = f"https://he.wikipedia.org/api/rest_v1/page/html/{query}"
    response = requests.get(heb_url)
    
    if response.status_code != 200:
        raise RuntimeError(f"Error fetching data for {name}. Response code: {response.status_code}")
    
    data = response.text
    
    birth_date = parse_birth_date_heb(data)
    death_date = parse_death_date_heb(data)

    if birth_date is not None and death_date is not None:
        return (birth_date, death_date)
    
    name = get_wikipedia_english_version(name)
    query = name.replace(" ", "_")
    eng_url = f"https://en.wikipedia.org/api/rest_v1/page/html/{query}"
    response = requests.get(eng_url)
    
    if response.status_code != 200:
        raise RuntimeError(f"Error fetching data for {name}. Response code: {response.status_code}")
    data = response.text
    
    #print(data)

    if birth_date is None:
        birth_date = parse_birth_date_eng(data)
    if death_date is None:
        death_date = parse_death_date_eng(data)
    
    return (birth_date, death_date)


# name = "אלברט איינשטיין"
# name = "עדה לאבלייס"
# name = "החפץ חיים"
# name = "נעמי שמר"
# name = "שמעון פרס"
# name = 'שמואל יוסף עגנון'
# print(get_birth_death_dates(name))


((1887, 8, 8), (1970, 2, 17))


In [94]:
import json
DB_FILE = "db.json"
NAMES_FILE = "names.txt"
with open(DB_FILE, "r", encoding='utf-8') as f:
    db = json.load(f)
for name, value in db.items():
    db[name] = tuple([tuple(x) for x in value])

with open(NAMES_FILE, "r") as f:
    names = f.read().splitlines()

for name in names:
    if name in db:
        continue
    print("Searching for name", name)
    try:
        birth_date, death_date = get_birth_death_dates(name)
        if birth_date is not None and death_date is not None:
            db[name] = (birth_date, death_date)
    except Exception as e:
        pass
with open(DB_FILE, "w", encoding='utf-8') as f:
    json.dump(db, f, indent=4)

db

{'אלברט איינשטיין': ((1879, 3, 14), (1955, 4, 18)),
 'עדה לאבלייס': ((1815, 12, 10), (1852, 11, 27)),
 'החפץ חיים': ((1839, 1, 26), (1933, 9, 15)),
 'נעמי שמר': ((1930, 7, 13), (2004, 6, 26)),
 'שמעון פרס': ((1923, 8, 2), (2016, 9, 28)),
 'שמואל יוסף עגנון': ((1887, 8, 8), (1970, 2, 17)),
 'משה מונטיפיורי': ((1784, 10, 24), (1885, 7, 28)),
 'חנה סנש': ((1921, 7, 17), (1944, 11, 7)),
 'נפוליאון בונפרטה': ((1769, 8, 15), (1821, 5, 5)),
 'קרל פרידריך גאוס': ((1777, 4, 30), (1855, 2, 23)),
 'אדולף היטלר': ((1889, 4, 20), (1945, 4, 30)),
 "וינסטון צ'רצ'יל": ((1874, 11, 30), (1965, 1, 24)),
 'אורסון ולס': ((1915, 5, 6), (1985, 10, 10))}

In [81]:
import json
DB_FILE = "db.json"
with open(DB_FILE, "w", encoding='utf-8') as f:
    json.dump(db, f, indent=4)


In [82]:
with open(DB_FILE, "r", encoding='utf-8') as f:
    db2 = json.load(f)

db2

{'אלברט איינשטיין': [[1879, 3, 14], [1955, 4, 18]],
 'עדה לאבלייס': [[1815, 12, 10], [1852, 11, 27]],
 'החפץ חיים': [[1839, 1, 26], [1933, 9, 15]],
 'נעמי שמר': [[1930, 7, 13], [2004, 6, 26]],
 'שמעון פרס': [[1923, 8, 2], [2016, 9, 28]],
 'שמואל יוסף עגנון': [[1887, 8, 8], [1970, 2, 17]],
 'משה מונטיפיורי': [[1784, 10, 24], [1885, 7, 28]]}

In [67]:
births = sorted([(name, val[0]) for name, val in db.items()], key=lambda x: x[1])
deaths = sorted([(name, val[1]) for name, val in db.items()], key=lambda x: x[1])

In [73]:
name = "אלברט איינשטיין"
num_to_find = 2
birth, death = db[name]
lived_when_old = []
lived_when_young = []

for name, birthday in reversed(births):
    if birthday < death:
        lived_when_old.append(name)
        if len(lived_when_old) == num_to_find:
            break

for name, deathday in deaths:
    if birth < deathday:
        lived_when_young.append(name)
        if len(lived_when_young) == num_to_find:
            break

print("Lived when young:")
print(lived_when_young)
print("Lived when old:")
print(lived_when_old)

Lived when young:
['משה מונטיפיורי', 'החפץ חיים']
Lived when old:
['נעמי שמר', 'שמעון פרס']


In [74]:
db

{'אלברט איינשטיין': ((1879, 3, 14), (1955, 4, 18)),
 'עדה לאבלייס': ((1815, 12, 10), (1852, 11, 27)),
 'החפץ חיים': ((1839, 1, 26), (1933, 9, 15)),
 'נעמי שמר': ((1930, 7, 13), (2004, 6, 26)),
 'שמעון פרס': ((1923, 8, 2), (2016, 9, 28)),
 'שמואל יוסף עגנון': ((1887, 8, 8), (1970, 2, 17)),
 'משה מונטיפיורי': ((1784, 10, 24), (1885, 7, 28))}

In [69]:

for name, birthday in reversed(births):
    if birthday < death:
        lived_when_old.append((name, birthday))
        if len(lived_when_old) == num_to_find:
            break
deaths

[('עדה לאבלייס', (1852, 11, 27)),
 ('משה מונטיפיורי', (1885, 7, 28)),
 ('החפץ חיים', (1933, 9, 15)),
 ('אלברט איינשטיין', (1955, 4, 18)),
 ('שמואל יוסף עגנון', (1970, 2, 17)),
 ('נעמי שמר', (2004, 6, 26)),
 ('שמעון פרס', (2016, 9, 28))]

In [13]:
import requests

def get_wikidata_birth_death(name):
    # SPARQL query to get birth and death dates from Wikidata
    query = """
    SELECT ?person ?personLabel ?birthDate ?deathDate WHERE {
      ?person wdt:P31 wd:Q5;           # Instance of human
              rdfs:label ?personLabel;  # Name of the person
              wdt:P569 ?birthDate.      # Birthdate
      OPTIONAL { ?person wdt:P570 ?deathDate }  # Death date (if applicable)
      FILTER(CONTAINS(LCASE(?personLabel), LCASE("%s")))
      FILTER(LANG(?personLabel) = "en")
    }
    LIMIT 1
    """ % name

    url = 'https://query.wikidata.org/sparql'
    headers = {
        'Accept': 'application/sparql-results+json'
    }
    params = {
        'query': query
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        return f"Error fetching data for {name}. Response code: {response.status_code}"

    data = response.json()

    if data['results']['bindings']:
        person_info = data['results']['bindings'][0]
        person_label = person_info['personLabel']['value']
        birth_date = person_info['birthDate']['value']
        death_date = person_info.get('deathDate', {}).get('value', "N/A")
        return f"{person_label}: Born on {birth_date}, Died on {death_date}"
    else:
        return f"Data not found for {name}."

# Example usage
name = "Albert Einstein"
print(get_wikidata_birth_death(name))


Error fetching data for Albert Einstein. Response code: 403
